In [1]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence, compute_bpc

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
from collections import deque
import random 
import pickle 
import os
import zipfile

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [3]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh', num_layers=2)
        self.linear = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded, h)
        out = self.linear(out[:,-1,:])

        return out, h  

In [4]:
# Step 1: Download and extract text8
def download_text8(path="dataset/text8.zip"):
    url = "http://mattmahoney.net/dc/text8.zip"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print("Downloading text8...")
        urllib.request.urlretrieve(url, path)
    with zipfile.ZipFile(path) as zf:
        data = zf.read(zf.namelist()[0]).decode('utf-8')
    return data

# Step 2: Build character-level vocabulary
def build_vocab(text):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

# Step 3: Encode text into integer tokens
def encode(text, stoi):
    return np.array([stoi[c] for c in text], dtype=np.int32)

# Step 4: Create input and target sequences
#def create_dataset(encoded_text, seq_len=64, step=1):
#    inputs, targets = [], []
#    for i in range(0, len(encoded_text) - seq_len, step):
#        inputs.append(encoded_text[i:i+seq_len])
#        targets.append(encoded_text[i+1:i+seq_len+1])
#    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

class Dataset_converter(Dataset):
    def __init__(self, encoded_text, working_memory=1, short_term_memory=8):
        
        self.X = []
        self.y = []

        for ii in range(0, len(encoded_text) - working_memory - short_term_memory, 1):
            self.X.append(
                encoded_text[ii:ii+short_term_memory]
            )
            self.y.append(
                encoded_text[ii+short_term_memory]
            )

        self.X = tnsr(np.array(self.X)).long()
        self.y = tnsr(np.array(self.y)).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [5]:
text = download_text8()
stoi, itos = build_vocab(text)
encoded = encode(text, stoi)
train_sample = 90000000
test_sample = 10000000
short_term_memory = 4

train_data_set = Dataset_converter(encoded[:train_sample], 1, short_term_memory=short_term_memory)
test_data_set = Dataset_converter(encoded[train_sample:], 1, short_term_memory=short_term_memory)

In [11]:
reps = 1
### initial training ###
working_memory = 1
total_layers = 6
# short_term_memory = 10
hidden_size = 256
vocab_size = 27
embedding_dim = 50
lr = 1e-4

for rep in range(reps):
    model = RNNEncoder(vocab_size, embedding_dim, hidden_size, num_layers=6)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    criterion = torch.nn.CrossEntropyLoss()
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10000)

    total = 0
    total_avg = 10000
    train_loader = DataLoader(train_data_set, batch_size=1, shuffle=False)
    bpc = 0
    acc = 0

    for X, y in train_loader:
        optimizer.zero_grad()

        if total == 0:
            y_pred, h = model(X)
        else:
            y_pred, h = model(X, hidden)

        loss = criterion(y_pred, y)     
        loss.backward()
        optimizer.step()

        # print(total)
        with torch.no_grad():
            hidden = h.detach()
            total += 1

            if y[0] == y_pred[0].argmax():
                    acc += 1
            
            bpc += compute_bpc(y_pred, y)
            
            
            if total%total_avg == 0:
                print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {acc/total_avg:.4f}, bpc: {bpc/total_avg:.4f}')
                acc = 0
                bpc = 0
                lr_scheduler.step()

Iter : 10001, loss: 2.4763, accuracy: 0.3151, bpc: 3.3798
Iter : 20001, loss: 0.6055, accuracy: 0.3914, bpc: 2.9956
Iter : 30001, loss: 3.5816, accuracy: 0.4181, bpc: 2.8166
Iter : 40001, loss: 2.6873, accuracy: 0.4314, bpc: 2.7628
Iter : 50001, loss: 1.8249, accuracy: 0.4310, bpc: 2.7287
Iter : 60001, loss: 2.9456, accuracy: 0.4609, bpc: 2.6082
Iter : 70001, loss: 0.5884, accuracy: 0.4900, bpc: 2.4981
Iter : 80001, loss: 0.3538, accuracy: 0.5104, bpc: 2.3913
Iter : 90001, loss: 1.0099, accuracy: 0.5074, bpc: 2.4336
Iter : 100001, loss: 3.1385, accuracy: 0.4874, bpc: 2.5038
Iter : 110001, loss: 1.4578, accuracy: 0.4916, bpc: 2.5037
Iter : 120001, loss: 1.1951, accuracy: 0.4751, bpc: 2.5639
Iter : 130001, loss: 2.4256, accuracy: 0.4786, bpc: 2.5236
Iter : 140001, loss: 0.9094, accuracy: 0.4777, bpc: 2.5218
Iter : 150001, loss: 2.5096, accuracy: 0.5043, bpc: 2.3964
Iter : 160001, loss: 1.7093, accuracy: 0.5573, bpc: 2.1537
Iter : 170001, loss: 0.9433, accuracy: 0.5291, bpc: 2.3036
Iter :

In [12]:
total = 0
bpc_test = 0
test_loader = DataLoader(test_data_set, batch_size=1, shuffle=False)
h = None 

for X, y in tqdm(test_loader):
    with torch.no_grad():
        logits, h = model(X, h) 

        bpc_test += compute_bpc(logits, y)

        total += 1
        
print(f'Finall BPC on test set: {bpc_test/total:.4f}')

100%|██████████| 9999995/9999995 [21:31<00:00, 7741.61it/s]

Finall BPC on test set: 2.0699
